# TF48_16T_4800Loop_Validation_Part2

This is the corrected TF48 branch of the Tomahawk confinement-screen validation.

## Controlling configuration

- 48 physical TF coils.
- 16 toroidal field periods.
- 100 turns per physical coil.
- 4,800 native loops total, all 16 periods manually pre-expanded into the coil files.
- 16 T target axis field.
- Native current: −125,000 A/turn.
- Strict direct edge-ripple gate: `<1.0%`.
- Native input header: `periods 1`, intentionally preventing a second automatic replication.

The bundled direct finite-segment preflight is 0.7539178029% at the outboard edge and the 16 T axis check is 16.000000099 T; both are input gates, not a native XGRID result. The prior screen reference of 0.594% is retained in the contract as a reference value.

The three files each contain 16 physical coils × 100 turns = 1,600 native loops. They are a runtime split only; together they contain the complete 48-coil/16-period geometry. No old 14 T field grid is used and no supplied field array is amplitude-rescaled.

## Kaggle run

Attach this package as the TF48 input dataset and attach the authorized `x-grid-zip` runtime dataset. Run these independently in three Kaggle sessions:

1. `notebooks/TF48_16T_4800Loop_Validation_Part1.ipynb`
2. `notebooks/TF48_16T_4800Loop_Validation_Part2.ipynb`
3. `notebooks/TF48_16T_4800Loop_Validation_Part3.ipynb`

Each notebook performs the full native XGRID run for its own 1,600-loop chunk, with no internal timeout by default and a 60-second heartbeat. It fails closed if the 16 T contract, hashes, pre-expanded loop count, grid, ripple gate, or native output is invalid.

The native executable is not fabricated or compiled by this package. The notebook searches the attached runtime dataset, `/home/STELLOPT/MAKEGRID/Release/xgrid`, PATH, and the configured KaggleHub dataset. Set `TF48_XGRID_TIMEOUT_SECONDS` only when an explicit internal cutoff is wanted; zero means unlimited.

## Local smoke test

From the extracted package directory, run:

    python3 tests/run_smoke_test.py

Smoke mode validates all input hashes, 48 physical coil identities, 16-period pre-expansion, all 4,800 native-loop records, native headers, direct axis/ripple preflight, and notebook execution. It does not produce a native MGRID and is not a native physics pass.

## Assembly

    python3 assembler/assemble_tf48_16t_4800.py --part1 TF48_PART1_OUTPUT.zip --part2 TF48_PART2_OUTPUT.zip --part3 TF48_PART3_OUTPUT.zip --output TF48_MGRID_16T_FULL_4800.nc

The assembler requires all three native worker reports to be green and verifies the assembled 16 T axis field and strict ripple gate. Free-boundary MHD, VMEC/DESC, and downstream qualification remain separate gates.


In [ ]:
#!/usr/bin/env python3
from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import shlex
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile
from pathlib import Path

try:
    import numpy as np
    from scipy.io import netcdf_file
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "numpy", "scipy"])
    import numpy as np
    from scipy.io import netcdf_file


BRANCH = "TF48"
PART_INDEX = 2
DATA_ENV = "TF48_DATA_ROOT"
WORK_ENV = "TF48_WORKDIR"
CONTRACT_NAME = "tf48_16t_4800_input_contract.json"
MANIFEST_NAME = "tf48_16t_4800_dataset_manifest.json"
COIL_FILE = "coils.tomahawk_tf48_4800_part2"
INPUT_FILE = "input_xgrid_tomahawk_tf48_4800_part2"
EXTCUR_FILE = "extcur.tomahawk_tf48_4800_part2"
EXPECTED_MGRID = "mgrid_tomahawk_tf48_4800_part2.nc"
EXPECTED_GROUPS = ['TF48']
EXPECTED_ALL_GROUPS = ['TF48']
EXPECTED_CURRENTS = [-125000.0]
EXPECTED_LOOP_COUNT = 1600
EXPECTED_FULL_LOOP_COUNT = 4800
EXPECTED_POINT_COUNT = 512
EXPECTED_PHYSICAL_COILS = 48
EXPECTED_FIELD_PERIODS = 16
TARGET_AXIS_FIELD_T = 16.0
TARGET_CURRENT_A = -125000.0
SHARED_GRID = {'rmin_m': 4.0, 'rmax_m': 11.0, 'zmin_m': -3.5, 'zmax_m': 3.5, 'toroidal_planes': 36, 'radial_points': 201, 'vertical_points': 201, 'mgrid_mode': 'S', 'stellarator_symmetry': False}

SMOKE = os.environ.get("TOMAHAWK_SMOKE_TEST", "").strip().lower() in {"1", "true", "yes", "on"}


class WorkerFailure(RuntimeError):
    pass


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def env_seconds(name: str, default: float) -> float:
    value = float(os.environ.get(name, str(default)).strip())
    if not math.isfinite(value) or value < 0.0:
        raise ValueError(f"{name} must be finite and non-negative")
    return value


def safe_extract(archive: Path, destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    base = destination.resolve()
    with zipfile.ZipFile(archive) as zfile:
        for info in zfile.infolist():
            target = (destination / info.filename).resolve()
            if target != base and base not in target.parents:
                raise RuntimeError(f"unsafe archive member: {info.filename}")
        zfile.extractall(destination)


def search_roots(work: Path) -> list[Path]:
    values = []
    if os.environ.get(DATA_ENV):
        values.append(Path(os.environ[DATA_ENV]).expanduser())
    here = Path.cwd().resolve()
    values.extend([here, here.parent, here.parent.parent, work, work.parent])
    values.extend(Path(item) for item in ("/kaggle/input", "/kaggle/working", "/kaggle/models", "/mnt/data"))
    result, seen = [], set()
    for value in values:
        try:
            resolved = value.resolve()
        except OSError:
            continue
        if resolved == Path("/") or not resolved.exists() or resolved in seen:
            continue
        result.append(resolved)
        seen.add(resolved)
    return result


def package_archives(roots: list[Path]) -> list[Path]:
    result, seen = [], set()
    for root in roots:
        if root.is_file() and root.suffix.lower() == ".zip":
            items = [root]
        elif root.is_dir():
            try:
                items = list(root.rglob("*.zip"))
            except OSError:
                items = []
        else:
            items = []
        for item in items:
            lower = item.name.lower()
            if BRANCH == "PF":
                accepted = "pf" in lower and "578" in lower
            else:
                accepted = "tf48" in lower and "16t" in lower and ("4800" in lower or "validation" in lower)
            if item not in seen and accepted:
                result.append(item)
                seen.add(item)
    return result


def validate_input_file(root: Path) -> list[str]:
    errors = []
    coil_path = root / COIL_FILE
    input_path = root / INPUT_FILE
    extcur_path = root / EXTCUR_FILE
    for path in (coil_path, input_path, extcur_path):
        if not path.is_file():
            errors.append(f"missing {path.name}")
    if errors:
        return errors
    lines = input_path.read_text(encoding="utf-8").splitlines()
    if len(lines) != 10:
        errors.append(f"{INPUT_FILE} has {len(lines)} lines, expected 10")
    else:
        expected = ["S", "N", "4", "11", "-3.5", "3.5", "36", "201", "201"]
        if lines[1:] != expected:
            errors.append(f"{INPUT_FILE} grid/mode lines do not match the shared grid")
    expected_stem = INPUT_FILE.removeprefix("input_xgrid_")
    if lines and lines[0].strip() != expected_stem:
        errors.append(f"{INPUT_FILE} first line is {lines[0]!r}, expected {expected_stem!r}")
    current_lines = [line for line in extcur_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    if BRANCH == "PF" and len(current_lines) != len(EXPECTED_GROUPS):
        errors.append(f"{EXTCUR_FILE} has {len(current_lines)} currents, expected {len(EXPECTED_GROUPS)}")
    if BRANCH == "TF48" and len(current_lines) != 1:
        errors.append(f"{EXTCUR_FILE} has {len(current_lines)} currents, expected one")
    if not coil_path.read_text(encoding="utf-8").startswith("periods 1\nbegin filament\nmirror NIL\n"):
        errors.append(f"{COIL_FILE} is not a pre-expanded periods-1 native coil file")
    return errors


def validate_root(root: Path) -> tuple[bool, list[str], dict]:
    errors = []
    contract_path = root / CONTRACT_NAME
    manifest_path = root / MANIFEST_NAME
    if not contract_path.is_file() or not manifest_path.is_file():
        return False, ["controlling contract or dataset manifest is missing"], {}
    try:
        contract = json.loads(contract_path.read_text(encoding="utf-8"))
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    except Exception as exc:
        return False, [f"invalid controlling JSON: {exc}"], {}
    if contract.get("branch") != BRANCH:
        errors.append(f"contract branch {contract.get('branch')!r} != {BRANCH!r}")
    grid = contract.get("shared_grid", {})
    for key, expected in SHARED_GRID.items():
        if grid.get(key) != expected:
            errors.append(f"shared_grid {key}={grid.get(key)!r}, expected {expected!r}")
    if BRANCH == "PF":
        if contract.get("full_loop_count") != EXPECTED_FULL_LOOP_COUNT:
            errors.append("PF full loop count is not 578")
        if contract.get("field_periods_nfp") != 1 or contract.get("native_periods_header") != 1:
            errors.append("PF native periodicity is not nfp=1")
        if contract.get("coil_groups") != EXPECTED_ALL_GROUPS:
            errors.append("PF full group list does not contain all six groups")
        chunk = contract.get("chunking", {}).get("part_definitions", {}).get(str(PART_INDEX), {})
        if chunk.get("groups") != EXPECTED_GROUPS or chunk.get("loop_count") != EXPECTED_LOOP_COUNT:
            errors.append("PF part definition does not match this worker")
    else:
        if contract.get("physical_coils") != EXPECTED_PHYSICAL_COILS:
            errors.append("TF48 physical coil count is not 48")
        if contract.get("field_periods") != EXPECTED_FIELD_PERIODS:
            errors.append("TF48 field-period count is not 16")
        if contract.get("target_axis_field_T") != TARGET_AXIS_FIELD_T:
            errors.append("TF48 target field is not 16 T")
        if contract.get("target_current_per_turn_A") != TARGET_CURRENT_A:
            errors.append("TF48 native current is not -125000 A/turn")
        if contract.get("full_loop_count") != EXPECTED_FULL_LOOP_COUNT:
            errors.append("TF48 full loop count is not 4,800")
        if contract.get("preexpanded_geometry") is not True or contract.get("native_periods_header") != 1:
            errors.append("TF48 geometry is not marked as manually pre-expanded with periods=1")
        chunk = contract.get("chunking", {}).get("part_definitions", {}).get(str(PART_INDEX), {})
        if chunk.get("physical_coils") != EXPECTED_PHYSICAL_COILS // 3 or chunk.get("loop_count") != EXPECTED_LOOP_COUNT:
            errors.append("TF48 part definition does not match this worker")
    for entry in manifest.get("files", []):
        path = root / entry.get("path", "")
        if not path.is_file():
            errors.append(f"manifest file missing: {entry.get('path')}")
        elif sha256(path) != entry.get("sha256"):
            errors.append(f"manifest SHA-256 mismatch: {entry.get('path')}")
    if manifest.get("contract_sha256") != sha256(contract_path):
        errors.append("contract SHA-256 mismatch")
    errors.extend(validate_input_file(root))
    return not errors, errors, contract


def find_data_root(work: Path) -> tuple[Path | None, list[str], dict]:
    roots = search_roots(work)
    candidates = []
    for root in roots:
        if root.is_dir():
            if (root / CONTRACT_NAME).is_file():
                candidates.append(root)
            try:
                candidates.extend(path.parent for path in root.rglob(CONTRACT_NAME))
            except OSError:
                pass
    extracted = work / "input_archives"
    extracted.mkdir(parents=True, exist_ok=True)
    for archive in package_archives(roots):
        destination = extracted / archive.stem
        marker = destination / ".extracted"
        if not marker.exists():
            try:
                safe_extract(archive, destination)
                marker.write_text("ok\n", encoding="utf-8")
            except Exception:
                continue
        try:
            candidates.extend(path.parent for path in destination.rglob(CONTRACT_NAME))
        except OSError:
            pass
    errors, seen = [], set()
    for candidate in candidates:
        try:
            candidate = candidate.resolve()
        except OSError:
            continue
        if candidate in seen:
            continue
        seen.add(candidate)
        valid, candidate_errors, contract = validate_root(candidate)
        if valid:
            return candidate, [], contract
        errors.extend([f"{candidate}: {error}" for error in candidate_errors])
    return None, errors[-80:], {}


def parse_native_coils(path: Path) -> dict:
    lines = path.read_text(encoding="utf-8").splitlines()
    loops = []
    points = []
    for raw in lines[3:-1]:
        fields = raw.split()
        if len(fields) == 4:
            points.append(tuple(float(value) for value in fields))
        elif len(fields) >= 6:
            if len(points) != EXPECTED_POINT_COUNT:
                raise WorkerFailure(f"{path.name}: loop has {len(points)} points, expected {EXPECTED_POINT_COUNT}")
            marker = tuple(float(value) for value in fields[:3])
            if max(abs(a - b) for a, b in zip(marker, points[0][:3])) > 1.0e-9:
                raise WorkerFailure(f"{path.name}: closure marker mismatch")
            loops.append({"group": fields[-1], "current_A": points[0][3]})
            points = []
        elif fields:
            raise WorkerFailure(f"{path.name}: malformed native line")
    if points:
        raise WorkerFailure(f"{path.name}: unterminated native loop")
    groups = [loop["group"] for loop in loops]
    if BRANCH == "PF":
        counts = {group: groups.count(group) for group in EXPECTED_GROUPS}
        if counts != {group: EXPECTED_FULL_COUNTS[group] for group in EXPECTED_GROUPS}:
            raise WorkerFailure(f"{path.name}: group counts {counts} do not match the part contract")
    else:
        if len(loops) != EXPECTED_LOOP_COUNT or set(groups) != {"TF48"}:
            raise WorkerFailure(f"{path.name}: loop count/group check failed")
    current_values = sorted({round(loop["current_A"], 6) for loop in loops})
    return {"loop_count": len(loops), "groups": groups, "current_values_A": current_values}


def find_xgrid(roots: list[Path], work: Path) -> str | None:
    candidates = []
    if os.environ.get("STELLOPT_XGRID"):
        candidates.append(os.environ["STELLOPT_XGRID"])
    candidates.extend(["/home/STELLOPT/MAKEGRID/Release/xgrid", "/usr/local/bin/xgrid", shutil.which("xgrid") or ""])
    for root in roots:
        if root.is_dir():
            try:
                candidates.extend(str(path) for path in root.rglob("xgrid") if path.is_file())
            except OSError:
                pass
    for candidate in candidates:
        try:
            parts = shlex.split(candidate)
        except ValueError:
            continue
        if len(parts) == 1 and Path(parts[0]).is_file():
            return str(Path(parts[0]).resolve())
        if len(parts) == 1 and shutil.which(parts[0]):
            return str(Path(shutil.which(parts[0])).resolve())
    if os.environ.get(f"{BRANCH}_DISABLE_KAGGLEHUB", "").strip().lower() in {"1", "true", "yes"}:
        return None
    handle = os.environ.get(f"{BRANCH}_XGRID_DATASET_HANDLE", "nicolebryant29/x-grid-zip").strip()
    if not handle:
        return None
    print(f"{BRANCH}_PART{PART_INDEX}_XGRID_DATASET_DOWNLOAD_ATTEMPT={handle}", flush=True)
    try:
        try:
            import kagglehub
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "kagglehub"])
            import kagglehub
        downloaded = Path(kagglehub.dataset_download(handle)).expanduser().resolve()
        for root in [downloaded]:
            if root.is_dir():
                try:
                    for path in root.rglob("xgrid"):
                        if path.is_file():
                            return str(path.resolve())
                except OSError:
                    pass
    except Exception as exc:
        print(f"{BRANCH}_PART{PART_INDEX}_XGRID_DATASET_DOWNLOAD_FAILED={exc!r}", flush=True)
    return None


def stage_xgrid(source: str, native_dir: Path) -> tuple[Path, dict]:
    source_path = Path(source)
    staged = native_dir / "xgrid"
    shutil.copy2(source_path, staged)
    staged.chmod(staged.stat().st_mode | 0o111)
    environment = os.environ.copy()
    library_paths = []
    for sibling in (source_path.parent / "lib", source_path.parent / "lib64"):
        if sibling.is_dir():
            destination = native_dir / sibling.name
            shutil.copytree(sibling, destination, dirs_exist_ok=True)
            library_paths.append(str(destination))
    if library_paths:
        old = environment.get("LD_LIBRARY_PATH", "")
        environment["LD_LIBRARY_PATH"] = os.pathsep.join(library_paths + ([old] if old else []))
    return staged, environment


def run_native(executable: Path, input_path: Path, native_dir: Path, log_path: Path, environment: dict) -> tuple[int, float, bool]:
    heartbeat = max(0.1, env_seconds(f"{BRANCH}_HEARTBEAT_SECONDS", 60.0))
    timeout = env_seconds(f"{BRANCH}_XGRID_TIMEOUT_SECONDS", 0.0)
    started = time.time()
    print(f"{BRANCH}_PART{PART_INDEX}_NATIVE_XGRID_START timeout={'UNLIMITED' if timeout == 0.0 else f'{timeout:.0f}s'}", flush=True)
    with input_path.open("r", encoding="utf-8") as stdin_file, log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            [str(executable)],
            stdin=stdin_file,
            cwd=native_dir,
            text=True,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            env=environment,
        )
        while True:
            try:
                code = process.wait(timeout=heartbeat)
                elapsed = time.time() - started
                print(f"{BRANCH}_PART{PART_INDEX}_NATIVE_XGRID_END return_code={code} elapsed_s={elapsed:.1f}", flush=True)
                return code, elapsed, False
            except subprocess.TimeoutExpired:
                elapsed = time.time() - started
                print(f"{BRANCH}_PART{PART_INDEX}_NATIVE_XGRID_RUNNING elapsed_s={elapsed:.1f} pid={process.pid}", flush=True)
                if timeout > 0.0 and elapsed >= timeout:
                    process.terminate()
                    try:
                        code = process.wait(timeout=30.0)
                    except subprocess.TimeoutExpired:
                        process.kill()
                        code = process.wait()
                    return code, time.time() - started, True


def chars_to_text(value: np.ndarray) -> str:
    pieces = []
    for item in np.asarray(value).reshape(-1):
        if isinstance(item, bytes):
            pieces.append(item.decode("ascii", errors="ignore"))
        else:
            pieces.append(str(item))
    return "".join(pieces).strip()


def decode_group_rows(value: np.ndarray) -> list[str]:
    array = np.asarray(value)
    rows = [array] if array.ndim == 1 else list(array)
    return [chars_to_text(row) for row in rows]


def inspect_mgrid(path: Path) -> dict:
    errors = []
    try:
        with netcdf_file(str(path), "r", mmap=False) as nc:
            dimensions = {key: int(value) for key, value in nc.dimensions.items() if value is not None}
            variables = {key: np.array(value.data).copy() for key, value in nc.variables.items()}
        for name, expected in {"phi": 36, "zee": 201, "rad": 201}.items():
            if dimensions.get(name) != expected:
                errors.append(f"dimension {name}={dimensions.get(name)!r}, expected {expected}")
        expected_external = len(EXPECTED_GROUPS) if BRANCH == "PF" else 1
        for name in ("external_coil_groups", "external_coils"):
            if dimensions.get(name) != expected_external:
                errors.append(f"dimension {name}={dimensions.get(name)!r}, expected {expected_external}")
        field_stats = {}
        for index in range(1, expected_external + 1):
            for component in ("br", "bp", "bz"):
                name = f"{component}_{index:03d}"
                if name not in variables or variables[name].shape != (36, 201, 201):
                    errors.append(f"missing or malformed {name}")
                elif not np.all(np.isfinite(variables[name])):
                    errors.append(f"non-finite values in {name}")
                else:
                    field_stats[name] = float(np.max(np.abs(variables[name])))
        if field_stats and max(field_stats.values()) == 0.0:
            errors.append("all native field arrays are zero")
        mode = chars_to_text(variables.get("mgrid_mode", np.asarray([])))
        if mode != "S":
            errors.append(f"mgrid_mode={mode!r}, expected S")
        nfp = int(round(float(np.asarray(variables.get("nfp", [0])).reshape(-1)[0]))) if "nfp" in variables else None
        if nfp != 1:
            errors.append(f"native MGRID nfp={nfp!r}, expected 1 for this branch")
        raw_current = np.asarray(variables.get("raw_coil_cur", np.asarray([])), dtype=float).reshape(-1)
        expected_currents = np.asarray(EXPECTED_CURRENTS, dtype=float)
        if raw_current.size != expected_currents.size or not np.allclose(raw_current, expected_currents, rtol=0.0, atol=1.0e-3):
            errors.append(f"raw_coil_cur={raw_current.tolist()}, expected {expected_currents.tolist()}")
        groups = decode_group_rows(variables.get("coil_group", np.asarray([])))
        if groups != EXPECTED_GROUPS:
            errors.append(f"coil_group={groups!r}, expected {EXPECTED_GROUPS!r}")
        for name, expected in (("ir", 201), ("jz", 201), ("kp", 36)):
            if name not in variables or int(round(float(np.asarray(variables[name]).reshape(-1)[0]))) != expected:
                errors.append(f"{name} metadata mismatch")
        for name, expected in (("rmin", 4.0), ("rmax", 11.0), ("zmin", -3.5), ("zmax", 3.5)):
            if name not in variables or not np.isclose(float(np.asarray(variables[name]).reshape(-1)[0]), expected, atol=1.0e-10):
                errors.append(f"{name} metadata mismatch")
        axis = None
        if BRANCH == "TF48" and "bp_001" in variables and not errors:
            axis = abs(float(variables["bp_001"][0, 100, 100]) * float(raw_current[0]))
        return {
            "path": str(path),
            "sha256": sha256(path),
            "dimensions": dimensions,
            "mgrid_mode": mode,
            "nfp": nfp,
            "coil_groups": groups,
            "raw_coil_cur_A": raw_current.tolist(),
            "field_abs_max_per_variable": field_stats,
            "axis_field_T": axis,
            "errors": errors,
            "valid": not errors,
        }
    except Exception as exc:
        return {"path": str(path), "errors": [f"MGRID read failed: {exc!r}"], "valid": False}


def compute_direct_tf_ripple(root: Path) -> dict:
    if BRANCH != "TF48":
        return {"status": "NOT_APPLICABLE_TO_PF"}
    rows = []
    path = root / "cad_a2_tf48_interleaved_centerlines.csv"
    with path.open(newline="", encoding="utf-8") as handle:
        for row in csv.DictReader(handle):
            rows.append([float(row[key]) for key in ("x_m", "y_m", "z_m", "next_x_m", "next_y_m", "next_z_m")])
    values = np.asarray(rows, dtype=float)
    p1, p2 = values[:, :3], values[:, 3:]
    mu0 = 4.0 * math.pi * 1.0e-7
    total_current = TARGET_CURRENT_A * 100.0

    def field_at(point: np.ndarray) -> np.ndarray:
        a = point - p1
        b = point - p2
        amag = np.linalg.norm(a, axis=1)
        bmag = np.linalg.norm(b, axis=1)
        cross = np.cross(a, b)
        denominator = amag * bmag * (amag * bmag + np.sum(a * b, axis=1))
        denominator = np.where(denominator == 0.0, np.inf, denominator)
        factor = (amag + bmag) / denominator
        return (mu0 * total_current / (4.0 * math.pi)) * (factor[:, None] * cross).sum(axis=0)

    axis = field_at(np.asarray([7.5, 0.0, 0.0]))
    samples = []
    for phi in np.linspace(0.0, 2.0 * math.pi / 48.0, 128):
        point = np.asarray([10.5 * math.cos(phi), 10.5 * math.sin(phi), 0.0])
        field = field_at(point)
        samples.append(float(field @ np.asarray([-math.sin(phi), math.cos(phi), 0.0])))
    values = np.asarray(samples)
    ripple = 100.0 * float(values.max() - values.min()) / float(values.max() + values.min())
    return {
        "axis_toroidal_field_T": float(axis[1]),
        "edge_ripple_percent": ripple,
        "ripple_limit_percent": 1.0,
        "strict_pass": bool(ripple < 1.0),
        "samples": 128,
        "user_reported_prior_full_grid_ripple_percent": 0.594,
    }


def make_output_zip(work: Path, data_root: Path | None, report_path: Path, native_dir: Path | None) -> Path:
    output = work / f"{BRANCH}_PART{PART_INDEX}_OUTPUT.zip"
    with zipfile.ZipFile(output, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zfile:
        if report_path.is_file():
            zfile.write(report_path, report_path.name)
        if data_root is not None and data_root.is_dir():
            for name in (CONTRACT_NAME, MANIFEST_NAME, COIL_FILE, INPUT_FILE, EXTCUR_FILE):
                path = data_root / name
                if path.is_file():
                    zfile.write(path, f"inputs/{path.name}")
        if native_dir is not None and native_dir.is_dir():
            for path in native_dir.iterdir():
                if path.is_file() and path.name != "xgrid" and not path.name.startswith(("coils.", "input_xgrid", "extcur.")):
                    zfile.write(path, f"native/{path.name}")
    return output


def main() -> None:
    started = time.time()
    if os.environ.get(WORK_ENV):
        work = Path(os.environ[WORK_ENV]).expanduser()
    elif Path("/kaggle/working").is_dir():
        work = Path("/kaggle/working") / f"{BRANCH.lower()}_part{PART_INDEX}_run"
    else:
        work = Path.cwd() / f"{BRANCH.lower()}_part{PART_INDEX}_run"
    work.mkdir(parents=True, exist_ok=True)
    report_path = work / f"{BRANCH}_PART{PART_INDEX}_REPORT.json"
    report = {
        "schema_version": "TOMAHAWK-CONFINEMENT-WORKER-1.0",
        "branch": BRANCH,
        "part": PART_INDEX,
        "smoke_mode": SMOKE,
        "status": "STARTED",
        "configuration": {
            "shared_grid": SHARED_GRID,
            "groups": EXPECTED_GROUPS,
            "full_groups": EXPECTED_ALL_GROUPS,
            "part_loop_count": EXPECTED_LOOP_COUNT,
            "full_loop_count": EXPECTED_FULL_LOOP_COUNT,
            "native_periods_header": 1,
            "physical_coils": EXPECTED_PHYSICAL_COILS,
            "field_periods": EXPECTED_FIELD_PERIODS,
            "target_axis_field_T": TARGET_AXIS_FIELD_T,
            "target_current_per_turn_A": TARGET_CURRENT_A,
        },
    }
    data_root = None
    native_dir = None
    try:
        data_root, locate_errors, contract = find_data_root(work)
        if data_root is None:
            report.update({"status": "INPUT_BLOCKED", "input_errors": locate_errors})
            report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n", encoding="utf-8")
            output = make_output_zip(work, None, report_path, None)
            print(f"{BRANCH}_PART{PART_INDEX}_OUTPUT_ZIP={output}")
            raise WorkerFailure("INPUT_BLOCKED: no verified branch data root found")
        valid, errors, contract = validate_root(data_root)
        report.update({"data_root": str(data_root), "input_manifest_pass": valid, "input_errors": errors})
        if not valid:
            report["status"] = "INPUT_BLOCKED"
            report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n", encoding="utf-8")
            output = make_output_zip(work, data_root, report_path, None)
            print(f"{BRANCH}_PART{PART_INDEX}_OUTPUT_ZIP={output}")
            raise WorkerFailure("INPUT_BLOCKED: manifest or native input validation failed")
        coil_summary = parse_native_coils(data_root / COIL_FILE)
        report["native_input_summary"] = {
            "coil_file": COIL_FILE,
            "loop_count": coil_summary["loop_count"],
            "groups": sorted(set(coil_summary["groups"])),
            "current_values_A": coil_summary["current_values_A"],
        }
        if BRANCH == "PF":
            report["native_input_summary"]["loops_by_group"] = {group: coil_summary["groups"].count(group) for group in EXPECTED_GROUPS}
        else:
            report["native_input_summary"].update({"physical_coils_in_part": EXPECTED_PHYSICAL_COILS // 3, "preexpanded_field_periods": 16})
            report["direct_ripple_preflight"] = compute_direct_tf_ripple(data_root)
            if not report["direct_ripple_preflight"]["strict_pass"]:
                report["status"] = "TF_RIPPLE_REJECTED"
                report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n", encoding="utf-8")
                output = make_output_zip(work, data_root, report_path, None)
                print(f"{BRANCH}_PART{PART_INDEX}_OUTPUT_ZIP={output}")
                raise WorkerFailure("TF_RIPPLE_REJECTED")
        if SMOKE:
            report.update({
                "status": "SMOKE_PASS",
                "native_execution": False,
                "native_solver_required_for_full_result": True,
                "accepted_for_assembly": False,
                "smoke_checks": [
                    "controlling contract hash",
                    "dataset manifest hashes",
                    "shared MGRID grid",
                    "native input header",
                    "native loop closure and counts",
                    "branch periodicity guard",
                ],
            })
            report["elapsed_s"] = time.time() - started
            report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n", encoding="utf-8")
            output = make_output_zip(work, data_root, report_path, None)
            print(f"{BRANCH}_PART{PART_INDEX}_SMOKE_STATUS=SMOKE_PASS")
            print(f"{BRANCH}_PART{PART_INDEX}_OUTPUT_ZIP={output}")
            return
        xgrid = find_xgrid(search_roots(work), work)
        if xgrid is None:
            report.update({"status": "NATIVE_XGRID_NOT_FOUND", "native_xgrid": None, "native_mgrid_present": False})
            report["elapsed_s"] = time.time() - started
            report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n", encoding="utf-8")
            output = make_output_zip(work, data_root, report_path, None)
            print(f"{BRANCH}_PART{PART_INDEX}_OUTPUT_ZIP={output}")
            raise WorkerFailure("NATIVE_XGRID_NOT_FOUND: attach the authorized x-grid-zip runtime dataset")
        native_dir = Path(tempfile.mkdtemp(prefix=f"{BRANCH.lower()}-part{PART_INDEX}-native-", dir=str(work)))
        for name in (COIL_FILE, INPUT_FILE, EXTCUR_FILE):
            shutil.copy2(data_root / name, native_dir / name)
        staged, environment = stage_xgrid(xgrid, native_dir)
        log_path = native_dir / f"xgrid_{BRANCH.lower()}_part{PART_INDEX}.log"
        code, elapsed, timed_out = run_native(staged, native_dir / INPUT_FILE, native_dir, log_path, environment)
        result_path = native_dir / EXPECTED_MGRID
        report.update({
            "native_xgrid": xgrid,
            "native_return_code": code,
            "native_elapsed_s": elapsed,
            "native_internal_timeout_enabled": env_seconds(f"{BRANCH}_XGRID_TIMEOUT_SECONDS", 0.0) > 0.0,
            "native_mgrid_present": result_path.is_file(),
            "native_log": str(log_path),
        })
        if timed_out:
            report["status"] = "NATIVE_XGRID_TIMEOUT"
        elif code == 0 and result_path.is_file():
            report["native_mgrid"] = inspect_mgrid(result_path)
            report["native_pass"] = bool(report["native_mgrid"]["valid"])
            report["status"] = "NATIVE_PART_PASS" if report["native_pass"] else "NATIVE_MGRID_INVALID"
        else:
            report["status"] = "NATIVE_XGRID_FAILED"
        report["accepted_for_assembly"] = bool(report.get("native_pass", False))
        report["elapsed_s"] = time.time() - started
        report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n", encoding="utf-8")
        output = make_output_zip(work, data_root, report_path, native_dir)
        print(f"{BRANCH}_PART{PART_INDEX}_STATUS={report['status']}")
        print(f"{BRANCH}_PART{PART_INDEX}_OUTPUT_ZIP={output}")
        if report["status"] != "NATIVE_PART_PASS":
            raise WorkerFailure(f"{BRANCH} Part {PART_INDEX} failed closed: {report['status']}")
    except WorkerFailure:
        raise
    except Exception as exc:
        report.update({"status": "RUNTIME_ERROR", "error": repr(exc), "elapsed_s": time.time() - started})
        report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n", encoding="utf-8")
        output = make_output_zip(work, data_root, report_path, native_dir)
        print(f"{BRANCH}_PART{PART_INDEX}_OUTPUT_ZIP={output}")
        raise


EXPECTED_FULL_COUNTS = {'TF48': 1600}
main()
